# Hierarchy loss on real data: the A/B for issue #67

*Tutorial — not exported, not run by `nbdev-test`. Needs your real multilabel data and a GPU.*

[`06_hierarchy_loss.ipynb`](06_hierarchy_loss.ipynb) showed the machinery on synthetic data. This notebook is the acceptance run:
train the **same** config twice (or more), differing **only** in `train.hierarchy.weight`, and measure on the val split,
on the **raw** predictions (before `apply_hierarchy`):

| Metric | Expected for B (weight > 0) vs A (weight 0) |
|---|---|
| violation rate `lifted ⊄ chip` | clearly lower |
| pixel IoU per class | `chip` / `void` not worse beyond seed spread; `lifted` equal or better |
| object recall / false alarms for `lifted` | not worse; fewer false alarms outside chip |
| change caused by `apply_hierarchy` | smaller for B |

**How to use:** set `RAW_DIR` / `PATCH_DIR` below, then run top to bottom. Runs are resumable (finished arms are skipped).
The last cell prints a markdown table to paste into the issue comment.

In [ ]:
# parameters — the only cell you edit
from pathlib import Path

REPO      = Path('..').resolve()                                   # run from tutorials/
SEMANTIC_SRC = '/home/hasan-spark/workspace/projects/data/universal_model/cegled_econo/lifted_chip/multimaterial_v2'
                                                                   # images/ + mask_semantic/ (0=bg, 1=chip, 2=lifted); set None if RAW_DIR is already per-class files
RAW_DIR   = REPO / 'outputs/hierarchy_ab/data'                     # images/ + masks/{id}_{class}.png (built below from SEMANTIC_SRC)
PATCH_DIR = REPO / 'outputs/hierarchy_ab/patches'                  # patch cache; REUSED by every arm = frozen train/val split
CONFIG    = 'configs/train_multilabel_3head.yaml'
EXTRA     = ['data.class_names=[chip,lifted]', 'train.num_classes=2', 'metrics.threshold=[0.5,0.5]',
             'patch.patch_size=512', 'patch.stride=512', 'train.crop_size=512', 'train.batch_size=16',
             'model.arch=resnet18', 'model.pretrained=true', 'model.weights_path=null',
             'train.freeze_epochs=0', 'train.epochs=6', 'train.resume=false', 'export.onnx=false']   # applied to every arm
WEIGHTS   = [0.0, 0.5]               # arm A first; add 0.1 / 1.0 to sweep
SEEDS     = [42, 43]                 # training seeds (split seed stays patch.seed=42 via the shared PATCH_DIR)
OUT_ROOT  = REPO / 'outputs/hierarchy_ab'
CLASSES   = ['chip', 'lifted']
CHIP, LIFTED = 0, 1
PAIRS     = [(LIFTED, CHIP)]         # lifted ⊂ chip

### Prepare the data

The source labels are one semantic map per image (`0` background, `1` chip, `2` lifted — lifted pixels *replace* the chip label).
The library's multilabel mode wants one binary mask per class, so we derive them: **chip = label ≥ 1** (lifted is part of the chip), **lifted = label 2**.
That makes `lifted ⊂ chip` true in the ground truth by construction, so any violation in the predictions is the model's. Images are symlinked; your source folder is not modified.

In [ ]:
import cv2, numpy as np

if SEMANTIC_SRC:
    src = Path(SEMANTIC_SRC); (RAW_DIR / 'images').mkdir(parents=True, exist_ok=True); (RAW_DIR / 'masks').mkdir(exist_ok=True)
    for f in sorted((src / 'images').glob('*.png')):
        link = RAW_DIR / 'images' / f.name
        if not link.exists(): link.symlink_to(f)
        if (RAW_DIR / 'masks' / f'{f.stem}_lifted.png').exists(): continue
        m = cv2.imread(str(src / 'mask_semantic' / f.name), cv2.IMREAD_UNCHANGED)
        cv2.imwrite(str(RAW_DIR / 'masks' / f'{f.stem}_chip.png'), ((m >= 1) * 255).astype(np.uint8))
        cv2.imwrite(str(RAW_DIR / 'masks' / f'{f.stem}_lifted.png'), ((m == 2) * 255).astype(np.uint8))
print(len(list((RAW_DIR / 'images').glob('*.png'))), 'images,', len(list((RAW_DIR / 'masks').glob('*.png'))), 'mask files')

## 1. Train the arms

One `segpt-train` call per (weight, seed). `hydra.run.dir` pins each run's output folder; the config is saved under `.hydra/` so the evaluation below reproduces the exact settings.

In [ ]:
import subprocess, sys

def run_dir(w, seed): return OUT_ROOT / f'w{w}_s{seed}'

def train_arm(w, seed):
    rd = run_dir(w, seed).resolve()
    if (rd / 'checkpoints' / 'best.pth').exists():
        print(f'skip w={w} seed={seed} (done)'); return
    cmd = ['segpt-train', str((REPO / CONFIG).resolve()), f'data.raw_dir={Path(RAW_DIR).resolve()}',
           f'data.patch_dir={Path(PATCH_DIR).resolve()}', f'train.hierarchy.weight={w}', f'train.seed={seed}',
           f'hydra.run.dir={rd}', 'hydra.job.chdir=true',          # chdir: each arm gets its own checkpoints/ folder
           *EXTRA]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

for seed in SEEDS:
    for w in WEIGHTS:
        train_arm(w, seed)

## 2. Evaluate on the val split — raw vs after `apply_hierarchy`

For every val patch: sigmoid → per-class threshold from the run's `metrics.threshold` → raw masks. Then measure violation, per-class IoU, and `lifted` object recall / false alarms,
once on the raw masks and once after `apply_hierarchy(raw, PAIRS)`.

In [ ]:
import importlib.util, numpy as np, torch
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

from segmentation_tools_pt.hydra_pipeline import resolve_channel_preset, resolve_mask_cfg, run_patch_pipeline
from segmentation_tools_pt.dataset import SegmentationDataset
from segmentation_tools_pt.transforms import get_val_transforms
from segmentation_tools_pt.metrics import subset_violation_rate, object_metrics
from segmentation_tools_pt.inference import apply_hierarchy

_spec = importlib.util.spec_from_file_location('segpt_train_script', REPO / 'scripts' / 'train.py')
_script = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_script)   # reuses the exact model-kwargs mapping

def _iou(p, g):
    u = (p | g).sum()
    return float((p & g).sum() / u) if u else 1.0

@torch.no_grad()
def evaluate(rd, device='cuda' if torch.cuda.is_available() else 'cpu'):
    cfg = OmegaConf.load(Path(rd) / '.hydra' / 'config.yaml')
    preset = resolve_channel_preset(cfg.data.data_mode)
    mask_cfg = resolve_mask_cfg(cfg)
    mean, std, root = run_patch_pipeline(cfg, Path(cfg.data.raw_dir), Path(cfg.data.patch_dir), preset['n_source_channels'],
                                         preset['channel_names'], cfg.data.mask_suffix or '')
    ds = SegmentationDataset(root, 'val', transform=get_val_transforms(cfg.train.crop_size, tuple(mean), tuple(std)),
                             num_classes=cfg.train.num_classes, n_source_channels=preset['n_source_channels'],
                             channel_names=preset['channel_names'], **mask_cfg)
    model = _script.get_model(**_script._model_kwargs(cfg, preset['in_channels'], pretrained=False))
    model.load_state_dict(torch.load(Path(rd) / 'checkpoints' / 'best.pth', map_location='cpu', weights_only=False)['model_state'])
    model.to(device).eval()
    thr = torch.tensor(OmegaConf.to_container(cfg.metrics.threshold), device=device).view(1, -1, 1, 1)
    out = dict(viol=[], iou_raw=[[] for _ in CLASSES], iou_clip=[[] for _ in CLASSES], clip_changed=[],
               obj=dict(raw=dict(total=0, detected=0, fa=0), clip=dict(total=0, detected=0, fa=0)))
    for b in DataLoader(ds, batch_size=16):
        raw = torch.sigmoid(model(b['image'].to(device))) > thr
        clip = apply_hierarchy(raw, PAIRS)
        gt = b['mask'].to(device).bool()
        for k in range(len(raw)):
            r, c, g = raw[k].cpu(), clip[k].cpu(), gt[k].cpu()
            out['viol'].append(subset_violation_rate(r[LIFTED].numpy(), r[CHIP].numpy()))
            out['clip_changed'].append(float((r != c).sum()))
            for ci in range(len(CLASSES)):
                out['iou_raw'][ci].append(_iou(r[ci], g[ci])); out['iou_clip'][ci].append(_iou(c[ci], g[ci]))
            for tag, m in (('raw', r), ('clip', c)):
                o = object_metrics(m[LIFTED].numpy(), g[LIFTED].numpy())
                out['obj'][tag]['total'] += o['total_gt']; out['obj'][tag]['detected'] += o['detected']; out['obj'][tag]['fa'] += o['false_alarms']
    return out

results = {(w, s): evaluate(run_dir(w, s)) for s in SEEDS for w in WEIGHTS}
print('evaluated', len(results), 'runs')

## 3. Results table (paste into the issue)

Per arm: mean ± std across seeds. `violation` is per-image, on **raw** predictions. `Δclip` is the number of pixels `apply_hierarchy` removes per patch — the smaller it is, the less the model relies on inference-time clipping.

In [ ]:
def ms(vals): return f'{np.mean(vals):.3f} ± {np.std(vals):.3f}' if len(vals) > 1 else f'{vals[0]:.3f}'

rows = []
for w in WEIGHTS:
    rs = [results[(w, s)] for s in SEEDS]
    rows.append({
        'weight': w,
        'violation (raw)': ms([np.mean(r['viol']) for r in rs]),
        **{f'IoU {n} (raw)': ms([np.mean(r['iou_raw'][i]) for r in rs]) for i, n in enumerate(CLASSES)},
        'lifted recall (raw)': ms([r['obj']['raw']['detected'] / max(r['obj']['raw']['total'], 1) for r in rs]),
        'lifted false alarms (raw)': ms([float(r['obj']['raw']['fa']) for r in rs]),
        'Δclip px/patch': ms([np.mean(r['clip_changed']) for r in rs]),
    })
cols = list(rows[0])
md = '| ' + ' | '.join(cols) + ' |\n|' + '---|' * len(cols) + '\n' + '\n'.join('| ' + ' | '.join(str(r[c]) for c in cols) + ' |' for r in rows)
print(md)

In [ ]:
# verdict per the acceptance rule: violation drops and no class loses more than the seed-to-seed spread
base = [results[(WEIGHTS[0], s)] for s in SEEDS]
for w in WEIGHTS[1:]:
    new = [results[(w, s)] for s in SEEDS]
    dv = np.mean([np.mean(r['viol']) for r in new]) - np.mean([np.mean(r['viol']) for r in base])
    print(f'weight {w}: Δviolation = {dv:+.4f}')
    for i, n in enumerate(CLASSES):
        a = [np.mean(r['iou_raw'][i]) for r in base]; b = [np.mean(r['iou_raw'][i]) for r in new]
        spread = max(np.std(a), np.std(b), 1e-3); d = np.mean(b) - np.mean(a)
        print(f'   IoU {n:7s}: {d:+.4f}  ({"within" if d > -spread else "BEYOND"} seed spread ±{spread:.4f})')

## 4. Look at it

The numbers can hide what the model does. Enable `monitoring.error_analysis` in the config (`ErrorAnalysisCallback`, writes worst-k posters at train end) for both arms and compare the worst-k `lifted` posters:
B should show fewer *lifted outside chip* blobs. To make the pairing exact, rerun with `EXTRA = ['monitoring.error_analysis.enabled=true']` (finished arms are skipped, so delete the run folder you want to redo).

**If it does not help:** a large weight switched on at epoch 0 can suppress a small head before it has learned anything (seen on the synthetic notebook). Try `0.1`, or a first loss phase with `w_hierarchy: 0` as warm-up, before concluding the loss is useless.

**Is the difference real?** Two seeds only bound the noise. When map #28's bootstrap CI / paired-comparison tooling lands, feed it the per-image `viol` / `iou_raw` lists collected above (`results[(w, seed)]`).